In [1]:
import os
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset


class CoronaryDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        # mask_path = os.path.join(self.mask_dir, self.images[idx].replace('.png', '_mask.png'))
        mask_path = os.path.join(self.mask_dir, self.images[idx])
        
        
        # print(img_path)
        # print(mask_path)

        # image = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        # if image is None:
        #     raise ValueError(f"Failed to load image: {img_path}")

        # image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
        # image = cv2.resize(image, (512, 512))

        # mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        # if mask is None:
        #     raise ValueError(f"Failed to load mask: {mask_path}")

        # mask = cv2.resize(mask, (512, 512))


        image = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
        image = cv2.resize(image, (512, 512))
        
        # mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        mask = cv2.cvtColor(mask, cv2.COLOR_BGRA2BGR)
        mask = cv2.resize(mask, (512, 512))
        
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        
        return image, mask

In [2]:
class CoronarySmallDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        # mask_path = os.path.join(self.mask_dir, self.images[idx].replace('.png', '_mask.png'))
        mask_path = os.path.join(self.mask_dir, self.images[idx])
        

        image = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2GRAY)
        image = cv2.resize(image, (256, 256))
        
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        mask = cv2.cvtColor(mask, cv2.COLOR_RGBA2GRAY)
        mask = cv2.resize(mask, (256, 256))
        
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        
        return image, mask

In [12]:
image_dir = 'images_train\input'
mask_dir = 'images_train\output'
images = os.listdir(image_dir)
for idx in range(1):
    img_path = os.path.join(image_dir, images[idx])
    mask_path = os.path.join(mask_dir, images[idx])

    image = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    # cv2.imshow('image', image)
    # cv2.imshow('mask', mask)

    # print("BGRA2BGR")
    # cv2.imshow('BGRA2BGR', cv2.cvtColor(mask, cv2.COLOR_BGRA2BGR))
    # print("RGBA2RGB")
    # cv2.imshow('RGBA2RGB', cv2.cvtColor(mask, cv2.COLOR_RGBA2RGB))
    # print("BGRA2GRAY")
    # cv2.imshow('BGRA2GRAY', cv2.cvtColor(mask, cv2.COLOR_BGRA2GRAY))
    # print("RGBA2GRAY")
    # cv2.imshow('RGBA2GRAY', cv2.cvtColor(mask, cv2.COLOR_RGBA2GRAY))

    image = cv2.resize(image, (256, 256))
    mask = cv2.resize(mask, (256, 256))
    # cv2.imshow('image', image)
    # cv2.imshow('mask', mask)

    image = cv2.cvtColor(image, cv2.COLOR_RGBA2GRAY)
    mask = cv2.cvtColor(mask, cv2.COLOR_RGBA2GRAY)
    cv2.imshow('image', image)
    cv2.imshow('mask', mask)
    cv2.waitKey(0)


(256, 256) (256, 256)


In [3]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision import transforms


transform = transforms.Compose([
    transforms.ToTensor()
])

train_image_dir = 'images_train\input'
train_mask_dir = 'images_train\output'
val_image_dir = 'images_val\input'
val_mask_dir = 'images_val\output'
test_image_dir = 'images_test\input'
test_mask_dir = 'images_test\output'

train_dataset = CoronarySmallDataset(train_image_dir, train_mask_dir, transform=transform)
val_dataset = CoronarySmallDataset(val_image_dir, val_mask_dir, transform=transform)
test_dataset = CoronarySmallDataset(test_image_dir, test_mask_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [4]:
from small_model import UNet


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet()
model = model.to(device)

In [5]:
import torch.optim as optim
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F


criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=50):
    best_loss = float('inf')
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for images, masks in tqdm(train_loader):
            images = images.to(device)
            masks = masks.to(device)

            # print(images.size())
            # print(images)
            # print(masks.size())
            # print(masks)

            optimizer.zero_grad()
            outputs = model(images)

            # print(outputs.size())
            # print(outputs)
            
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
        
        train_loss = train_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)

                outputs = model(images)
                loss = criterion(outputs, masks)
                
                val_loss += loss.item() * images.size(0)
        
        val_loss = val_loss / len(val_loader.dataset)
        
        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), 'model.pth')


In [6]:
train_model(model, train_loader, val_loader, criterion, optimizer)

  0%|          | 0/61 [00:00<?, ?it/s]

100%|██████████| 61/61 [01:39<00:00,  1.63s/it]


Epoch 1/50, Train Loss: 0.6459, Val Loss: 0.6245


100%|██████████| 61/61 [01:43<00:00,  1.69s/it]


Epoch 2/50, Train Loss: 0.6057, Val Loss: 0.6103


...

Epoch 20/50, Train Loss: 0.5795, Val Loss: 0.5951


100%|██████████| 61/61 [02:00<00:00,  1.98s/it]


Epoch 21/50, Train Loss: 0.5793, Val Loss: 0.5949


  8%|▊         | 5/61 [00:10<01:54,  2.05s/it]


KeyboardInterrupt: 

In [21]:

model.load_state_dict(torch.load('model.pth'))
model.eval()

def show_image(type, image_name):
    dir = f"images_test\{type}"
    print(dir)
    img = cv2.imread(os.path.join(dir, image_name), cv2.IMREAD_UNCHANGED)
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2GRAY)
    img = cv2.resize(img, (256, 256))
    cv2.imshow(type, img)

def predict(image_name):
    dir = 'images_test\input'
    img = cv2.imread(os.path.join(dir, image_name), cv2.IMREAD_UNCHANGED)
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2GRAY)
    img = cv2.resize(img, (256, 256))
    img = transforms.ToTensor()(img).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(img)
        pred = pred.squeeze().cpu().numpy()
        print(pred)
        # pred = (pred > 0.5).astype(np.uint8)
    
    cv2.imshow('pred', pred)
    cv2.waitKey(0)
   

In [22]:
show_image("input", "13c2ur549vohc0jat2dvt3xrmw2_26.png")
show_image("output", "13c2ur549vohc0jat2dvt3xrmw2_26.png")
predict("13c2ur549vohc0jat2dvt3xrmw2_26.png")

images_test\input
images_test\output
[[0.40213355 0.3343382  0.34606147 ... 0.35704365 0.4147155  0.46797222]
 [0.31283566 0.2805955  0.28972852 ... 0.2919705  0.3223169  0.39079577]
 [0.33817658 0.32035193 0.31999618 ... 0.31690672 0.34919697 0.37780985]
 ...
 [0.37762988 0.33748683 0.34643078 ... 0.6266077  0.5825866  0.51315796]
 [0.39910752 0.36674517 0.3465057  ... 0.5873193  0.5589406  0.54267055]
 [0.433026   0.39033005 0.36490428 ... 0.58684075 0.56157666 0.54411906]]


In [ ]:
model.load_state_dict(torch.load('model.pth'))

def dice_coefficient(y_true, y_pred):
    smooth = 1.0
    y_true_f = y_true.view(-1)
    y_pred_f = y_pred.view(-1)
    intersection = (y_true_f * y_pred_f).sum()
    return (2. * intersection + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

model.eval()
test_loss = 0.0
dice_scores = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)
        
        test_loss += loss.item() * images.size(0)
        dice_scores.append(dice_coefficient(masks, outputs))

test_loss = test_loss / len(test_loader.dataset)
mean_dice = torch.mean(torch.tensor(dice_scores))

print(f'Test Loss: {test_loss:.4f}')
print(f'Mean Dice Coefficient: {mean_dice:.4f}')


In [ ]:
from skimage import morphology

def post_process(prediction):
    prediction_np = prediction.cpu().numpy()
    processed = morphology.remove_small_objects(prediction_np > 0.5, min_size=100)
    processed = morphology.remove_small_holes(processed, area_threshold=100)
    return torch.tensor(processed, device=device)

post_processed_predictions = [post_process(pred) for pred in outputs]


In [ ]:
from flask import Flask, request, jsonify
import io

app = Flask(__name__)
model.load_state_dict(torch.load('model.pth'))
model.eval()

@app.route('/predict', methods=['POST'])
def predict():
    file = request.files['image']
    img_bytes = file.read()
    img = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_UNCHANGED)
    img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    img = cv2.resize(img, (512, 512))
    img = transforms.ToTensor()(img).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(img)
        pred = pred.squeeze().cpu().numpy()
        pred = (pred > 0.5).astype(np.uint8)
    
    _, buffer = cv2.imencode('.png', pred)
    response = io.BytesIO(buffer)
    response.seek(0)
    return response, 200, {'Content-Type': 'image/png'}

if __name__ == '__main__':
    app.run()
